In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
INPUT_BASE = Path('/content/drive/MyDrive/Research')
BASE = Path('/content/drive/MyDrive/version_2')
DRIVE_ROOT = Path('/content/drive/MyDrive')


def resolve_file(*patterns):
    search_roots = [INPUT_BASE, BASE, DRIVE_ROOT]
    seen = set()
    for root in search_roots:
        if root in seen or not root.exists():
            continue
        seen.add(root)
        for pattern in patterns:
            matches = sorted(root.rglob(pattern))
            if matches:
                print(f'Resolved {pattern}: {matches[0]}')
                return matches[0]
    raise FileNotFoundError(patterns)


def stage_local_file(src_path, cache_dir=Path('/content/ds340w_cache')):
    cache_dir.mkdir(parents=True, exist_ok=True)
    dst_path = cache_dir / Path(src_path).name
    try:
        same_size = dst_path.exists() and dst_path.stat().st_size == Path(src_path).stat().st_size
    except OSError:
        same_size = False
    if same_size:
        print(f'Using cached local file: {dst_path}')
        return dst_path
    print(f'Copying to local runtime storage: {src_path} -> {dst_path}')
    shutil.copy2(src_path, dst_path)
    print(f'Local copy ready: {dst_path}')
    return dst_path


DATA_PATH = resolve_file('Updated_Crime_Enriched_PointLevel_v16c.csv', '*v16c*.csv')
LOCAL_DATA_PATH = stage_local_file(DATA_PATH)
OUT_DIR = BASE / 'paper_data_tables_v16c'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_PATH =', DATA_PATH)
print('LOCAL_DATA_PATH =', LOCAL_DATA_PATH)
print('OUT_DIR =', OUT_DIR)

## Load Full Dataset

In [ ]:
df = pd.read_csv(LOCAL_DATA_PATH, low_memory=False)
print('Loaded full dataset shape:', df.shape)

In [ ]:
for col in ['crime_date', 'occurred_dt']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

if 'row_id' in df.columns:
    df['row_id_numeric'] = pd.to_numeric(df['row_id'], errors='coerce')

for col in ['latitude', 'longitude', 'crime_precinct', 'addr_pct_cd']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

## Table 1: Final Validated v16c Dataset Summary

In [ ]:
table_1_rows = [
    {'Statistic': 'Total rows', 'Value': int(df.shape[0])},
    {'Statistic': 'Total columns', 'Value': int(df.shape[1])},
]

if 'crime_date' in df.columns:
    table_1_rows.append({'Statistic': 'Crime date start', 'Value': str(df['crime_date'].min())[:10]})
    table_1_rows.append({'Statistic': 'Crime date end', 'Value': str(df['crime_date'].max())[:10]})

if 'row_id_numeric' in df.columns:
    table_1_rows.append({'Statistic': 'Placeholder rows (row_id = 0)', 'Value': int((df['row_id_numeric'] == 0).fillna(False).sum())})

for col, label in [
    ('cmplnt_fr_tm_raw', 'Non-missing complaint raw time'),
    ('cmplnt_fr_tm', 'Non-missing complaint time'),
    ('occurred_dt', 'Non-missing occurred_dt'),
    ('crime_time', 'Non-missing crime_time'),
    ('latitude', 'Non-missing latitude'),
    ('longitude', 'Non-missing longitude')
]:
    if col in df.columns:
        table_1_rows.append({'Statistic': label, 'Value': int(df[col].notna().sum())})

table_1 = pd.DataFrame(table_1_rows)
table_1

## Build The Same Point-Level Modeling Subset Used By The Notebook

In [ ]:
usecols = [
    'row_id', 'crime_date', 'crime_time', 'occurred_dt', 'cmplnt_fr_dt', 'cmplnt_fr_tm',
    'ofns_desc', 'crime_precinct', 'addr_pct_cd', 'latitude', 'longitude',
    'holiday', 'weekend', 'is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday',
    't_mean', 'rh_mean', 'wind_mean', 'has_event', 'event_match_count', 'has_road_closure', 'road_closure_count'
]

raw = pd.read_csv(LOCAL_DATA_PATH, usecols=lambda c: c in usecols, low_memory=False)
raw_shape_before = raw.shape

if 'row_id' in raw.columns:
    raw = raw.loc[pd.to_numeric(raw['row_id'], errors='coerce').fillna(-1) != 0].copy()

def parse_datetime(df_in):
    if 'occurred_dt' in df_in.columns:
        dt = pd.to_datetime(df_in['occurred_dt'], errors='coerce', utc=True)
        if dt.notna().any():
            return dt.dt.tz_convert(None)
    if 'crime_date' in df_in.columns and 'crime_time' in df_in.columns:
        dt = pd.to_datetime(df_in['crime_date'].astype(str) + ' ' + df_in['crime_time'].astype(str), errors='coerce', utc=True)
        if dt.notna().any():
            return dt.dt.tz_convert(None)
    if 'cmplnt_fr_dt' in df_in.columns and 'cmplnt_fr_tm' in df_in.columns:
        dt = pd.to_datetime(df_in['cmplnt_fr_dt'].astype(str) + ' ' + df_in['cmplnt_fr_tm'].astype(str), errors='coerce', utc=True)
        return dt.dt.tz_convert(None)
    raise ValueError('No usable datetime columns found.')

raw['dt'] = parse_datetime(raw)
raw_after_placeholder = raw.shape
raw = raw.dropna(subset=['dt', 'latitude', 'longitude']).copy()
raw_after_valid_dt_coords = raw.shape

print('Loaded modeling subset shape:', raw_shape_before)
print('After dropping placeholder rows:', raw_after_placeholder)
print('After keeping valid datetime and coordinates:', raw_after_valid_dt_coords)

## Table 2: Point-Level Modeling Reduction Summary

In [ ]:
table_2 = pd.DataFrame([
    {'Component': 'Full validated v16c dataset', 'Rows': int(df.shape[0]), 'Columns': int(df.shape[1]), 'Notes': 'Final cleaned incident-level working dataset'},
    {'Component': 'Loaded subset for point-level notebook', 'Rows': int(raw_shape_before[0]), 'Columns': int(raw_shape_before[1]), 'Notes': 'Only columns required by the point-level notebook were loaded'},
    {'Component': 'After dropping placeholder rows', 'Rows': int(raw_after_placeholder[0]), 'Columns': int(raw_after_placeholder[1]), 'Notes': 'Removed row_id = 0 placeholder record'},
    {'Component': 'After valid datetime/coordinate filtering', 'Rows': int(raw_after_valid_dt_coords[0]), 'Columns': int(raw_after_valid_dt_coords[1]), 'Notes': 'Retained only incidents with usable time and coordinates'}
])
table_2

## Table 3: Compact Feature Schema For The Point-Level Model

In [ ]:
schema_rows = [
    {
        'Feature group': 'Identifiers and crime descriptors',
        'Variables': 'row_id, ofns_desc, crime_precinct, addr_pct_cd',
        'Role in pipeline': 'Record tracking, crime labeling, and precinct context'
    },
    {
        'Feature group': 'Temporal fields',
        'Variables': 'crime_date, crime_time, occurred_dt, cmplnt_fr_dt, cmplnt_fr_tm',
        'Role in pipeline': 'Timestamp construction and hourly aggregation'
    },
    {
        'Feature group': 'Spatial fields',
        'Variables': 'latitude, longitude',
        'Role in pipeline': 'Projection into NYC space and assignment to grid cells'
    },
    {
        'Feature group': 'Weather features',
        'Variables': 't_mean, rh_mean, wind_mean',
        'Role in pipeline': 'External environmental context'
    },
    {
        'Feature group': 'Calendar features',
        'Variables': 'holiday, weekend, is_christian_holiday, is_islamic_holiday, is_jewish_holiday, is_hindu_holiday',
        'Role in pipeline': 'Holiday and routine-activity context'
    },
    {
        'Feature group': 'Event and closure features',
        'Variables': 'has_event, event_match_count, has_road_closure, road_closure_count',
        'Role in pipeline': 'Contextual event and closure information'
    }
]

table_3 = pd.DataFrame(schema_rows)
table_3

## Save Tables To Drive

In [ ]:
table_1.to_csv(OUT_DIR / 'table_1_final_v16c_dataset_summary.csv', index=False)
table_2.to_csv(OUT_DIR / 'table_2_pointlevel_modeling_reduction_summary.csv', index=False)
table_3.to_csv(OUT_DIR / 'table_3_pointlevel_feature_schema.csv', index=False)

manifest = {
    'dataset_file': str(DATA_PATH),
    'output_dir': str(OUT_DIR),
    'saved_tables': [
        'table_1_final_v16c_dataset_summary.csv',
        'table_2_pointlevel_modeling_reduction_summary.csv',
        'table_3_pointlevel_feature_schema.csv'
    ]
}
(OUT_DIR / 'saved_tables_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Saved tables to', OUT_DIR)
for name in manifest['saved_tables']:
    print('-', name)

## Recommendation For The Paper

In [ ]:
print('Use Table 1 in Methodology subsection IV.C.')
print('Use Table 2 in Methodology subsection IV.D.')
print('Use Table 3 only if you have space; otherwise merge its ideas into the text around Table 2.')